In [ ]:
import torch 
import torch.nn.functional as F 
import matplotlib.pyplot as plt 
%matplotlib inline 


In [ ]:
words = open("names (1).txt" ,"r").read().splitlines()
words[:8]


In [ ]:
len(words)

In [ ]:
chars= sorted(list(set(''.join(words))))
mapping  = {c : i+1  for i,c in enumerate(chars)}
mapping['.'] = 0
reverse_mapping = {i:c for c,i in mapping.items()}
reverse_mapping


In [ ]:
block_size = 3
X , y = [] , []
for w in words[:5]:
    print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = mapping[ch]
        X.append(context)
        y.append(ix)
        print(''.join(reverse_mapping[i] for i in context) ,'---->' , reverse_mapping[ix])
        context = context[1:] + [ix]
        
X=torch.tensor(X)
y=torch.tensor(y)

In [ ]:
X.shape , y.shape

In [ ]:
C=torch.randn((27,2))


In [ ]:
F.one_hot(torch.tensor(5) , num_classes= 27).float() @ C

In [ ]:
#embedding X for 27 by 2 indexing
emb = C[X]


In [ ]:
w1 = torch.rand((6 ,100))
b = torch.rand(100)

In [ ]:
torch.cat([emb[: , 0 , :] , emb[: , 1 , :] , emb[: , 2 , :] ] , 1).shape

In [ ]:
#this work when we change the block size 
torch.cat(torch.unbind(emb , 1) , 1).shape

In [ ]:
h = torch.tanh(emb.view(-1 , 6) @ w1 + b)
h


In [ ]:
#creating the final layer or output layer
W2 = torch.randn((100 , 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2  + b2

In [ ]:
logits

In [ ]:
counts = logits.exp()

In [ ]:
prob = counts / counts.sum(1 , keepdim=True)
prob[0].sum()

In [ ]:
prob[torch.arange(32) , y ]

In [ ]:
loss =  - prob[torch.arange(32) , y ].log().mean()
loss

In [ ]:
g=torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2) , generator=g)
W1= torch.randn((6,100) , generator=g)
b1 = torch.rand(100 , generator=g)
W2 = torch.randn((100 , 27) , generator=g)
b2 = torch.randn(27 , generator=g)

parameters = [C,W1,W2,b2]

In [ ]:
emb = C[X]
h = torch.tanh(emb.view(-1 , 6) @ w1 + b)
logits = h @ W2  + b2
counts = logits.exp()
prob = counts / counts.sum(1 , keepdim=True)
loss =  - prob[torch.arange(32) , y ].log().mean()
loss

In [ ]:
#loss using crossentropy
F.cross_entropy(logits , y)

In [ ]:
for p in parameters:
    p.requires_grad = True

In [50]:
for _ in range(20):
    #forward pass 
    emb = C[X]
    h = torch.tanh(emb.view(-1 , 6) @ W1 + b1)
    logits = h @ W2  + b2
    loss =F.cross_entropy(logits , y)
    print(loss.item())
    #backward pass 
    for p in parameters:
        p.grad= None
    loss.backward()

    #update 
    for p in parameters:
        p.data += - 0.1 * p.grad 

18.177806854248047
15.684249877929688
14.293636322021484
12.995829582214355
11.395089149475098
10.383330345153809
9.492012977600098
8.680302619934082
7.942598342895508
7.297031879425049
6.736738204956055
6.253318786621094
5.832248210906982
5.459559440612793
5.120577812194824
4.805299282073975
4.508674144744873
4.228358745574951
3.9634413719177246
3.713709831237793
